# 05 — PaliGemma 2 3B Fine-Tuning: Document Classification
**Group 1 – Invoices | Step 5**

| # | Model | Parametri | Strategia |
|---|-------|-----------|-----------|
| 1 | **PaliGemma 2 3B** | ~3B | LoRA (r=16) + classification head |

**Strategy:** Two-phase fine-tuning with LoRA on RVL-CDIP (16 classes).  
Direct classification via logits — no generate() call, much faster than Qwen.  
Optimised for RTX Pro 6000 Blackwell (96GB VRAM), bf16.


## 0 — Environment setup

In [ ]:
import os

def _is_kaggle():
    return os.path.exists('/kaggle/working')
def _is_colab():
    try:
        import google.colab
        return not _is_kaggle()
    except ImportError:
        return False
PLATFORM = 'kaggle' if _is_kaggle() else ('colab' if _is_colab() else 'local')
print(PLATFORM)


In [ ]:
if PLATFORM == 'colab':
    USE_DRIVE = True
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = '/content/drive/MyDrive/NLP_Invoices/outputs'
    else:
        OUTPUT_DIR = '/content/outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
if PLATFORM == 'kaggle':
    OUTPUT_DIR = '/kaggle/working/outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    HF_TOKEN = None
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN loaded from Kaggle Secrets')
    except Exception:
        print('HF_TOKEN not found')


In [ ]:
if PLATFORM == 'local':
    OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    from huggingface_hub import get_token, login
    HF_TOKEN = get_token() or os.environ.get('HF_TOKEN', '')
    if not HF_TOKEN: raise ValueError('HF_TOKEN not found.')
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Login HuggingFace OK')


## 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q datasets transformers torchvision tqdm
!{sys.executable} -m pip install -q accelerate peft
print('✅ Dependencies installed')


## 1 — Imports & configuration

In [ ]:
import random, gc, pickle, time, warnings
from collections import defaultdict, Counter
from io import BytesIO
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, PeftModel
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score
)
from tqdm.notebook import tqdm

# ── Iperparametri ──────────────────────────────────────────────────────────────
MODEL_ID        = "google/paligemma2-3b-pt-224"
N_TRAIN_CLASS   = 2000
N_VAL_CLASS     = 100
INVOICE_LABEL   = 6
RANDOM_SEED     = 42
NUM_WORKERS     = 4
BATCH_SIZE      = 32      # direct logits → large batch, much faster than Qwen
IMG_SIZE        = 224

# LoRA
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

# Fase 1: classification head only (backbone frozen)
LR_HEAD         = 3e-4
EPOCHS_HEAD     = 3

# Phase 2: LoRA on vision encoder + LLM + head
LR_FULL         = 5e-5
EPOCHS_FULL     = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

LABEL_NAMES = {
    0:"advertisement", 1:"budget",   2:"email",         3:"file folder",
    4:"form",          5:"handwritten", 6:"invoice",     7:"letter",
    8:"memo",          9:"news article", 10:"presentation", 11:"questionnaire",
    12:"resume",       13:"scientific publication", 14:"scientific report",
    15:"specification"
}
ALL_LABELS    = list(LABEL_NAMES.keys())
CLASS_NAMES   = [LABEL_NAMES[i] for i in ALL_LABELS]
NUM_CLASSES   = len(CLASS_NAMES)

BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "paligemma2_lora_best")
print("Config ready.")


## 2 — Helper functions

In [ ]:
def to_pil(image_field):
    if isinstance(image_field, Image.Image):
        return image_field.convert("RGB")
    elif isinstance(image_field, dict):
        return Image.open(BytesIO(image_field["bytes"])).convert("RGB")
    else:
        return Image.fromarray(np.array(image_field)).convert("RGB")

print("Helper functions ready.")


## 3a — Cache dataset to disk
Reuses the same cache as the DeiT notebook if it exists.


In [ ]:
CACHE_DIR  = Path(OUTPUT_DIR) / "dataset_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / f"rvl_cdip_train{N_TRAIN_CLASS}_val{N_VAL_CLASS}_testFULL_seed{RANDOM_SEED}.pkl"

def download_and_cache():
    print("Scarico dataset via streaming...\n")
    data = {}
    for split_name, hf_split, n, seed in [
        ("train", "train",      N_TRAIN_CLASS, RANDOM_SEED),
        ("val",   "validation", N_VAL_CLASS,   RANDOM_SEED + 1),
        ("test",  "test",       None,          RANDOM_SEED + 2),
    ]:
        print(f"Caricamento {split_name}...")
        stream = load_dataset("chainyo/rvl-cdip", split=hf_split, streaming=True
                  ).shuffle(seed=seed, buffer_size=20_000)
        bucket, done, skipped = defaultdict(list), set(), 0
        for ex in iter(stream):
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    label = int(ex["label"])
                    if n is not None and label in done:
                        continue
                    img = to_pil(ex["image"])
                    buf = BytesIO()
                    img.save(buf, format="PNG")
                    bucket[label].append({"image_bytes": buf.getvalue(), "label": label})
                    if n is not None and len(bucket[label]) == n:
                        done.add(label)
                        print(f"  ✓ {LABEL_NAMES[label]}")
                        if len(done) == len(ALL_LABELS):
                            break
            except Exception:
                skipped += 1
        if n is None:
            for lbl in ALL_LABELS:
                print(f"  ✓ {LABEL_NAMES[lbl]}: {len(bucket[lbl])}")
        if skipped:
            print(f"  ⚠️  {skipped} corrupted images skipped")
        examples = [e for exs in bucket.values() for e in exs]
        random.seed(seed)
        random.shuffle(examples)
        data[split_name] = examples
        print(f"✅ {split_name}: {len(examples)} examples\n")
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(data, f, protocol=4)
    print(f"💾 Cache salvata → {CACHE_FILE}")
    return data

if CACHE_FILE.exists():
    print(f"✅ Cache trovata: {CACHE_FILE.name}")
    with open(CACHE_FILE, "rb") as f:
        dataset = pickle.load(f)
else:
    dataset = download_and_cache()

train_examples = dataset["train"]
val_examples   = dataset["val"]
test_examples  = dataset["test"]
true_labels    = [ex["label"] for ex in test_examples]

print(f"Train: {len(train_examples)}  Val: {len(val_examples)}  Test: {len(test_examples)}")
test_dist = Counter(ex["label"] for ex in test_examples)
for lbl in ALL_LABELS:
    print(f"  {LABEL_NAMES[lbl]:<30} {test_dist.get(lbl,0):>4}")


## 3b — Supplement missing classes in the test set

In [ ]:
MISSING_LABELS = [l for l in ALL_LABELS if Counter(ex["label"] for ex in test_examples).get(l, 0) == 0]
print(f"Missing classes: {[LABEL_NAMES[l] for l in MISSING_LABELS]}")

if MISSING_LABELS:
    NAME_TO_LABEL = {v: k for k, v in LABEL_NAMES.items()}
    fill_stream = load_dataset("vaclavpechtor/rvl_cdip-small-200", split="train", streaming=True)
    ext_label_names = fill_stream.features["label"].names
    EXT_TO_MINE = {i: NAME_TO_LABEL.get(name) for i, name in enumerate(ext_label_names)}
    fill_bucket, fill_done = defaultdict(list), set()

    for ex in fill_stream:
        my_label = EXT_TO_MINE.get(int(ex["label"]))
        if my_label is None or my_label not in MISSING_LABELS or my_label in fill_done:
            continue
        img = to_pil(ex["image"])
        buf = BytesIO()
        img.save(buf, format="PNG")
        fill_bucket[my_label].append({"image_bytes": buf.getvalue(), "label": my_label})
        if len(fill_bucket[my_label]) >= 100:
            fill_done.add(my_label)
            print(f"  ✓ {LABEL_NAMES[my_label]}: 100 examples")
        if fill_done == set(MISSING_LABELS):
            break

    for exs in fill_bucket.values():
        test_examples.extend(exs)
    true_labels = [ex["label"] for ex in test_examples]
    print(f"\n✅ Test: {len(test_examples)} totali")
    print(f"   Invoice examples: {sum(1 for ex in test_examples if ex['label'] == INVOICE_LABEL)}")
else:
    print("✅ No missing classes.")


## 4 — Load PaliGemma 2 3B & classification head
PaliGemma2 is used as a **vision encoder + feature extractor**.  
We attach a linear classification head on top of the pooled output.  
No generate() call — direct classification via logits, like DeiT.


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Load the PaliGemma2 backbone
backbone = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# ── Wrapper with classification head ─────────────────────────────────────────
class PaliGemmaClassifier(nn.Module):
    def __init__(self, backbone, num_classes, hidden_size=None):
        super().__init__()
        self.backbone = backbone
        # PaliGemma hidden size: 2048 for the 3B model
        if hidden_size is None:
            hidden_size = backbone.config.text_config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, pixel_values, input_ids, attention_mask, labels=None):
        outputs = self.backbone(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # Use the last hidden state of the first token as features
        hidden = outputs.hidden_states[-1][:, 0, :].to(torch.float32)
        logits = self.classifier(hidden)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return loss, logits

model = PaliGemmaClassifier(backbone, NUM_CLASSES).to(DEVICE)

# ── LoRA on the backbone ──────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model.backbone = get_peft_model(model.backbone, lora_config)
model.backbone.print_trainable_parameters()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParametri totali   : {total/1e6:.1f}M")
print(f"Parametri trainable: {trainable/1e6:.1f}M  ({100*trainable/total:.2f}%)")
print(f"VRAM after load    : {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Model ready.")


## 5 — Dataset & DataLoader

In [ ]:
# Minimal prompt required by PaliGemma to activate the vision encoder
PALIGEMMA_PROMPT = "<image> classify document"

class PaliGemmaDocDataset(Dataset):
    def __init__(self, examples, augment=False):
        self.examples = examples
        if augment:
            from torchvision import transforms
            self.aug = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.3),
                transforms.RandomRotation(degrees=5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            ])
        else:
            self.aug = None

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        ex = self.examples[i]
        try:
            img = Image.open(BytesIO(ex["image_bytes"])).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), 0)
        if self.aug:
            img = self.aug(img)
        return img, int(ex["label"])


def collate_fn(batch):
    images, labels = zip(*batch)
    inputs = processor(
        text=[PALIGEMMA_PROMPT] * len(images),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    return inputs, torch.tensor(labels)


def make_loader(examples, augment=False, shuffle=False):
    ds = PaliGemmaDocDataset(examples, augment=augment)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_fn)

print("Dataset utilities ready.")


## 6 — Training utilities

In [ ]:
def freeze_backbone(model):
    """Freeze the backbone — Phase 1: only the classification head is trainable."""
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 1 — trainable: {n/1e6:.3f}M (head only)")

def unfreeze_lora(model):
    """Unfreeze the backbone LoRA weights — Phase 2: LoRA + head."""
    for name, param in model.backbone.named_parameters():
        if "lora_" in name:
            param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Fase 2 — trainable: {n/1e6:.2f}M (LoRA + head)")

# Class weights — upweight invoice class
class_weights = torch.ones(NUM_CLASSES, device=DEVICE)
class_weights[INVOICE_LABEL] = 8.0
criterion = nn.CrossEntropyLoss(weight=class_weights.to(torch.float32), label_smoothing=0.1)
print(f"Invoice class weight: {class_weights[INVOICE_LABEL].item():.1f}x")

def run_epoch(model, loader, optimizer=None, phase="train"):
    is_train = phase == "train"
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with (torch.enable_grad() if is_train else torch.no_grad()):
        for inputs, labels in loader:
            pixel_values  = inputs["pixel_values"].to(DEVICE)
            input_ids     = inputs["input_ids"].to(DEVICE)
            attention_mask = inputs["attention_mask"].to(DEVICE)
            labels        = labels.to(DEVICE)

            loss, logits = model(pixel_values, input_ids, attention_mask, labels)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            preds    = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return total_loss / total, correct / total

def train_phase(model, train_loader, val_loader, optimizer, scheduler,
                epochs, phase_name, best_val_loss):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, "train")
        vl_loss, vl_acc = run_epoch(model, val_loader,   None,      "val")
        scheduler.step()
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)
        print(f"  [{phase_name}] Epoch {epoch:02d}/{epochs}"
              f"  loss {tr_loss:.4f}/{vl_loss:.4f}"
              f"  acc {tr_acc:.3f}/{vl_acc:.3f}"
              f"  ({time.time()-t0:.0f}s)")
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            torch.save(model.state_dict(), BEST_MODEL_PATH + ".pt")
            print(f"    💾 Best model salvato (val_loss={vl_loss:.4f})")
    return history, best_val_loss

print("Training utilities ready.")


## 7 — Phase 1: classification head only (backbone frozen)

In [ ]:
freeze_backbone(model)

train_loader = make_loader(train_examples, augment=True,  shuffle=True)
val_loader   = make_loader(val_examples,   augment=False, shuffle=False)

optimizer1 = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=LR_HEAD, weight_decay=1e-4)
scheduler1 = CosineAnnealingLR(optimizer1, T_max=EPOCHS_HEAD, eta_min=1e-6)

print(f"\nFase 1 — {EPOCHS_HEAD} epoch, LR={LR_HEAD}")
best_val_loss = float("inf")
history1, best_val_loss = train_phase(
    model, train_loader, val_loader, optimizer1, scheduler1, EPOCHS_HEAD, "HEAD", best_val_loss)
print("\n✅ Fase 1 completata.")


## 8 — Phase 2: LoRA + classification head

In [ ]:
unfreeze_lora(model)

optimizer2 = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=LR_FULL, weight_decay=1e-4)
scheduler2 = CosineAnnealingLR(optimizer2, T_max=EPOCHS_FULL, eta_min=1e-7)

print(f"\nFase 2 — {EPOCHS_FULL} epoch, LR={LR_FULL}")
history2, best_val_loss = train_phase(
    model, train_loader, val_loader, optimizer2, scheduler2, EPOCHS_FULL, "FULL", best_val_loss)
print("\n✅ Fase 2 completata.")
print(f"Best val loss: {best_val_loss:.4f}")


## 9 — Learning curves

In [ ]:
all_train_loss = history1["train_loss"] + history2["train_loss"]
all_val_loss   = history1["val_loss"]   + history2["val_loss"]
all_train_acc  = history1["train_acc"]  + history2["train_acc"]
all_val_acc    = history1["val_acc"]    + history2["val_acc"]
epochs_total   = len(all_train_loss)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, tv, vv, ylabel in [
    (ax1, all_train_loss, all_val_loss, "Loss"),
    (ax2, all_train_acc,  all_val_acc,  "Accuracy"),
]:
    xs = range(1, epochs_total + 1)
    ax.plot(xs, tv, label="Train", marker="o")
    ax.plot(xs, vv, label="Val",   marker="s")
    ax.axvline(EPOCHS_HEAD + 0.5, color="gray", linestyle="--", linewidth=1, label="Fase 2")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel); ax.legend()
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("PaliGemma 2 3B LoRA — Learning Curves", fontsize=13)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "05_paligemma_learning_curves.png")
fig.savefig(path, dpi=150); plt.show()
print(f"Saved → {path}")


## 10 — Evaluation on test set

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH + ".pt", map_location=DEVICE))
model.eval()
print(f"✅ Best model loaded")

test_loader = make_loader(test_examples, augment=False, shuffle=False)
all_preds, all_true = [], []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Test inference"):
        pixel_values   = inputs["pixel_values"].to(DEVICE)
        input_ids      = inputs["input_ids"].to(DEVICE)
        attention_mask = inputs["attention_mask"].to(DEVICE)
        _, logits = model(pixel_values, input_ids, attention_mask)
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_true.extend(labels.tolist())

acc      = accuracy_score(all_true, all_preds)
f1_macro = f1_score(all_true, all_preds, average="macro", zero_division=0)
inv_true = [1 if l == INVOICE_LABEL else 0 for l in all_true]
inv_pred = [1 if p == INVOICE_LABEL else 0 for p in all_preds]
inv_f1   = f1_score(inv_true, inv_pred, zero_division=0)

print(f"\n{'─'*55}")
print(f"  PaliGemma 2 3B (LoRA fine-tuned)")
print(f"{'─'*55}")
print(f"  Overall Accuracy : {acc:.3f}")
print(f"  Macro F1         : {f1_macro:.3f}")
print(f"  Invoice F1       : {inv_f1:.3f}\n")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES,
                             labels=ALL_LABELS, digits=3, zero_division=0))

fig, ax = plt.subplots(figsize=(14, 11))
cm = confusion_matrix(all_true, all_preds, labels=ALL_LABELS)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, colorbar=False, cmap="Blues", xticks_rotation=45)
ax.set_title(f"PaliGemma 2 3B LoRA — Confusion Matrix  (acc={acc:.3f})", fontsize=12, pad=14)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "05_paligemma_confusion_matrix.png")
fig.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved → {path}")


## 11 — Save results

In [ ]:
results_path = os.path.join(OUTPUT_DIR, "05_paligemma_results.txt")
with open(results_path, "w") as f:
    f.write("PaliGemma 2 3B LoRA Fine-Tuning — Results\n")
    f.write("=" * 55 + "\n\n")
    f.write(f"Model    : {MODEL_ID}\n")
    f.write(f"Dataset  : chainyo/rvl-cdip\n")
    f.write(f"Train    : {N_TRAIN_CLASS}/class x {NUM_CLASSES} = {len(train_examples)}\n")
    f.write(f"Val      : {N_VAL_CLASS}/class x {NUM_CLASSES} = {len(val_examples)}\n")
    f.write(f"Test     : {len(test_examples)} examples total\n")
    f.write(f"Device   : {DEVICE}\n\n")
    f.write(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}\n")
    f.write(f"Fase 1   : {EPOCHS_HEAD} epoch, LR={LR_HEAD}, backbone frozen\n")
    f.write(f"Fase 2   : {EPOCHS_FULL} epoch, LR={LR_FULL}, LoRA + head\n\n")
    f.write(f"Overall Accuracy : {acc:.3f}\n")
    f.write(f"Macro F1         : {f1_macro:.3f}\n")
    f.write(f"Invoice F1       : {inv_f1:.3f}\n\n")
    f.write(classification_report(all_true, all_preds, target_names=CLASS_NAMES,
                                  labels=ALL_LABELS, digits=3, zero_division=0))
print(f"Results saved → {results_path}")
print("\n✅ Step 5 (PaliGemma 2 3B LoRA) complete.")


## 12 — Comparison with previous models

In [ ]:
def load_metrics(path):
    acc, f1, inv = None, None, None
    if os.path.isfile(path):
        with open(path) as f:
            for line in f:
                if "Overall Accuracy" in line: acc = float(line.split(":")[-1])
                if "Macro F1"         in line: f1  = float(line.split(":")[-1])
                if "Invoice F1"       in line: inv = float(line.split(":")[-1])
    return acc, f1, inv

ALL_RESULTS = {"PaliGemma2-3B LoRA": {"accuracy": acc, "f1_macro": f1_macro, "inv_f1": inv_f1}}

for name, fname in [
    ("DeiT-small FT",       "03_deit_results.txt"),
    ("Qwen2.5-VL-3B LoRA",  "04_qwen25vl_results.txt"),
]:
    a, f, i = load_metrics(os.path.join(OUTPUT_DIR, fname))
    if a is not None:
        ALL_RESULTS[name] = {"accuracy": a, "f1_macro": f, "inv_f1": i}
        print(f"✅ {name}: acc={a:.3f} inv_f1={i:.3f}")

model_names = list(ALL_RESULTS.keys())
accs_p = [ALL_RESULTS[m]["accuracy"]  for m in model_names]
f1s_p  = [ALL_RESULTS[m]["f1_macro"]  for m in model_names]
invs_p = [ALL_RESULTS[m]["inv_f1"]    for m in model_names]

COLOURS = ["#e74c3c", "#3498db", "#27ae60", "#9b59b6"]
x, w = np.arange(len(model_names)), 0.25
fig, ax = plt.subplots(figsize=(max(10, len(model_names)*3), 6))
b1 = ax.bar(x-w, accs_p, w, label="Accuracy",   color=COLOURS[:len(model_names)], alpha=0.6)
b2 = ax.bar(x,   f1s_p,  w, label="Macro F1",   color=COLOURS[:len(model_names)])
b3 = ax.bar(x+w, invs_p, w, label="Invoice F1", color=COLOURS[:len(model_names)], alpha=0.35, hatch="//")

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f"{h:.2f}", ha="center", fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0, 1.15); ax.set_ylabel("Score")
ax.set_title("Model Comparison — RVL-CDIP Document Classification", fontsize=12)
ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "05_comparison_all_models.png")
fig.savefig(path, dpi=150); plt.show()
print(f"Saved → {path}")

print("\n── Summary ─────────────────────────────────────────────────")
print(f"{'Model':<25} {'Accuracy':>10} {'Macro F1':>10} {'Invoice F1':>12}")
print("─" * 62)
best_acc = max(accs_p); best_inv = max(invs_p)
for m in model_names:
    r = ALL_RESULTS[m]
    tag_a = "★" if r["accuracy"] == best_acc else " "
    tag_i = "★" if r["inv_f1"]   == best_inv else " "
    print(f"{m:<25} {r['accuracy']:>10.3f}{tag_a} {r['f1_macro']:>10.3f}  {r['inv_f1']:>10.3f}{tag_i}")
print("  ★ = best")
